# Magic Hour Subtitles — Phase 1 Smoke Test

This notebook installs the pushed Phase 1 pipeline, uploads one MP4, runs `run_pipeline()`, validates the result, and downloads `output.mp4`. Use a Colab GPU runtime.

In [ ]:
# 1. Runtime checks
import shutil
import subprocess

def command_version(command, args):
    executable = shutil.which(command)
    if not executable:
        return f"{command}: NOT FOUND"
    result = subprocess.run(
        [executable, *args], capture_output=True, text=True, check=False
    )
    text = (result.stdout or result.stderr).splitlines()
    return text[0] if text else f"{command}: available"

try:
    import torch
    print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch is not available for the initial CUDA check.")

print(command_version("nvidia-smi", ["--query-gpu=name,driver_version", "--format=csv,noheader"]))
print(command_version("ffmpeg", ["-version"]))
print(command_version("ffprobe", ["-version"]))

In [ ]:
# 2. Clone and install Phase 1
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/rahulsiiitm/magic-hour-subtitles.git"
REPO_DIR = Path("/content/magic-hour-subtitles")
PROJECT_DIR = REPO_DIR / "KillerSubtitles-main"

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR), "--no-deps"], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

import ctranslate2
CT2_CUDA_DEVICES = ctranslate2.get_cuda_device_count()
print(f"CTranslate2 CUDA devices: {CT2_CUDA_DEVICES}")
if CT2_CUDA_DEVICES == 0:
    print("⚠️ faster-whisper will fall back to the smaller CPU/int8 model.")
else:
    print("✓ faster-whisper CUDA path is available.")

In [ ]:
# 3. Upload exactly one MP4
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one MP4 file.")

uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if Path(uploaded_name).suffix.lower() != ".mp4":
    raise ValueError("The uploaded file must have an .mp4 extension.")

INPUT_PATH = Path("/content/input.mp4")
INPUT_PATH.write_bytes(uploaded_bytes)
print(f"Uploaded: {uploaded_name} -> {INPUT_PATH} ({INPUT_PATH.stat().st_size / 1_048_576:.2f} MB)")

In [ ]:
# 4. Run the existing Phase 1 pipeline
import warnings

from killer_subtitles.ffmpeg import get_video_info
from killer_subtitles.models import LayoutConfig, PipelineConfig, StyleConfig
from killer_subtitles.pipeline import run_pipeline

OUTPUT_PATH = Path("/content/output.mp4")
if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()

video_info = get_video_info(INPUT_PATH)
font_path = PROJECT_DIR / "killer_subtitles" / "fonts" / "Montserrat-ExtraBold.ttf"
config = PipelineConfig(
    input_video=INPUT_PATH,
    output_video=OUTPUT_PATH,
    style=StyleConfig(
        font_path=str(font_path),
        font_size=max(1, int(video_info.height * 0.05)),
    ),
    layout=LayoutConfig(
        mode="karaoke",
        position="lower",
        margin_x=int(video_info.width * 0.10),
        margin_y=int(video_info.height * 0.05),
    ),
)

fallback = {"used": False}
original_showwarning = warnings.showwarning

def showwarning(message, category, filename, lineno, file=None, line=None):
    if issubclass(category, RuntimeWarning) and "CPU" in str(message):
        fallback["used"] = True
        print(f"⚠️ FASTER-WHISPER CPU FALLBACK: {message}")
    else:
        original_showwarning(message, category, filename, lineno, file, line)

def progress(phase, current, total):
    if current == 0:
        print(f"→ {phase}")

warnings.showwarning = showwarning
try:
    result = run_pipeline(config, video_info=video_info, progress_callback=progress)
finally:
    warnings.showwarning = original_showwarning

if not fallback["used"]:
    print("✓ No faster-whisper CPU fallback warning was emitted.")
print(f"Generated: {result}")

In [ ]:
# 5. Validate source and output media
import json
from fractions import Fraction

def probe_media(path):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error", "-show_streams",
            "-show_format", "-of", "json", str(path),
        ],
        capture_output=True, text=True, check=True,
    )
    data = json.loads(result.stdout)
    streams = data.get("streams", [])
    video = next((s for s in streams if s.get("codec_type") == "video"), None)
    rate = (video or {}).get("avg_frame_rate", "0/1")
    fps = float(Fraction(rate)) if rate not in {"0/0", "N/A"} else 0.0
    duration = float(data.get("format", {}).get("duration", 0.0))
    return {
        "duration": duration,
        "fps": fps,
        "size_mb": Path(path).stat().st_size / 1_048_576,
        "video": video is not None,
        "audio": any(s.get("codec_type") == "audio" for s in streams),
    }

source_stats = probe_media(INPUT_PATH)
output_stats = probe_media(OUTPUT_PATH)
print(f"{'Media':<10} {'Duration':>10} {'FPS':>8} {'Size MB':>10} {'Video':>8} {'Audio':>8}")
for label, stats in (("Source", source_stats), ("Output", output_stats)):
    print(
        f"{label:<10} {stats['duration']:>9.3f}s {stats['fps']:>8.3f} "
        f"{stats['size_mb']:>10.2f} {str(stats['video']):>8} {str(stats['audio']):>8}"
    )

In [ ]:
# 6. Preview and download output.mp4
from IPython.display import Video, display

display(Video(str(OUTPUT_PATH), embed=True, width=720))
files.download(str(OUTPUT_PATH))